# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DevOwais-ai/FlyRank-Week1_Assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Answer

* **Chosen Lane:** Lane 2 — Refresh / Content Opportunity Scoring[cite: 1]
* **ML Task Formulation:** Supervised Binary Classification / Learning to Rank[cite: 1]
* **Task Description:** Predict the probability that a given piece of content is in a state of decay or underperformance ($P(\text{declining}) \in [0, 1]$)[cite: 1].
* **Actionable Output:** A prioritized, score-ranked review queue where each content item receives a score (0–100) along with explicit, rule-backed reason codes (e.g., `declining_with_demand`, `stale_visible_page`) to guide human editorial review[cite: 1].

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Answer

* **Proxy Label Used:** `is_declining_label = (trend_direction == 'down')`[cite: 1].
* **Target Definition:** A binary classification variable ($y \in \{0, 1\}$) where `1` indicates that a content item is experiencing sustained traffic or visibility decline, and `0` indicates stable or growing performance[cite: 1].
* **Current Proxy vs. Ideal Future Target:**
  * **Current Proxy (Starter Dataset):** Uses `trend_direction == 'down'` calculated over the current 90-day snapshot[cite: 1]. While effective for initial exploration, it is a current-window heuristic rather than a true future prediction[cite: 1].
  * **Ideal Target (Full Warehouse):** A leakage-safe temporal window design where observed signals from a prior window (e.g., prior 90 days) predict measurable traffic decay or recovery over a distinct future window (e.g., next 30 days)[cite: 1].

## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Answer

* **Primary Metric:** **Precision@K** (specifically Precision@20 and Precision@50)[cite: 1].
  * **Why Precision@K:** Editorial teams have fixed human bandwidth (e.g., reviewing 20 to 50 pages per week)[cite: 1]. This metric measures what percentage of the top $K$ recommended pages in the prioritized queue are truly declining and worth reviewing[cite: 1].
* **Secondary Evaluation Metrics:**
  * **Average Precision (PR-AUC):** Evaluates overall ranking performance across the entire precision-recall curve without being skewed by class imbalance[cite: 1].
  * **ROC AUC:** Measures the model's baseline ability to separate declining pages from stable pages across all threshold cutoffs[cite: 1].
* **Business Metric Alignment:** Maximizing Precision@50 minimizes wasted editorial time on false positives (pages that did not need a refresh), ensuring high ROI on editorial labor hours[cite: 1].

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

###Answer

* **Grain (Unit of Analysis):** Exactly one row represents a single pseudonymized content item / page (`content_id` / `content_hash_id`)[cite: 1].
* **Feature Window Context:** Aggregate performance metrics evaluated over a prior 90-day window[cite: 1].

In [2]:
import os
import pandas as pd

# 1. Locate and load the dataset safely
repo_path = 'FlyRank-Week1_Assignment1'
if not os.path.exists(repo_path) and not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    os.system(f"git clone https://github.com/DevOwais-ai/FlyRank-Week1_Assignment1.git {repo_path}")

possible_paths = [
    '../data/raw/content_refresh_anonymized.csv',
    'data/raw/content_refresh_anonymized.csv',
    f'{repo_path}/data/raw/content_refresh_anonymized.csv'
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)
df = pd.read_csv(data_path)

# 2. Sketch the target column (Proxy Label)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# 3. Display unit of analysis inspection
print(f"Dataframe Shape: {df.shape} (Rows = Content Items, Columns = Features/Labels)")
print(f"Unique Content Items: {df['content_id'].nunique():,}\n")

# Display key columns including the unit of analysis and target label
display_cols = [
    'content_id',
    'client_id',
    'impressions_90d',
    'sessions_90d',
    'content_age_days',
    'trend_direction',
    'is_declining_label'
]

df[display_cols].head(5)

Dataframe Shape: (30000, 45) (Rows = Content Items, Columns = Features/Labels)
Unique Content Items: 30,000



,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Answer

* **Multi-Signal Interaction:** Fixed heuristic rules (such as `if trend == 'down' and impressions >= 500`) evaluate columns in isolated silos. A machine learning model learns non-linear interactions across high-dimensional signals simultaneously—combining subtle position drift, content age, word count tiers, and engagement drops.
* **Continuous Probabilities vs. Arbitrary Cliffs:** Hard threshold cutoffs create artificial boundary errors (e.g., treating 499 impressions completely differently from 500)[cite: 1]. ML outputs continuous probabilities $P(\text{declining}) \in [0, 1]$, enabling a granular, smooth prioritization queue[cite: 1].
* **Superior Top-K Precision:** Empirical baseline tests in `outputs/model_report.md` demonstrate that learned models (e.g., Random Forest) achieve a **Precision@50 of ~0.740**, significantly outperforming static rule baselines (**Precision@50 of ~0.240**)[cite: 1]. This directly saves editorial bandwidth by ensuring top recommendations are accurate[cite: 1].

## Self-check

- [x] **ML Task Type Named:** Defined as Supervised Binary Classification / Learning to Rank[cite: 1].
- [x] **Target / Proxy Defined:** Set `is_declining_label = (trend_direction == 'down')` as the proxy label[cite: 1].
- [x] **Success Metric Selected:** Selected Precision@K (Precision@20 / Precision@50) aligned with editorial review capacity[cite: 1].
- [x] **Unit of Analysis Displayed:** Loaded dataset and demonstrated that 1 row = 1 unique content item (`content_id`)[cite: 1].
- [x] **ML vs. Rule Advantage Explained:** Detailed why probabilistic multi-signal learning outperforms fixed heuristic thresholds[cite: 1].